# Structured — Retrospective Dataset — Cleaning

Produces:
- `structured_behavioral_clean.csv` (task lifecycle events)
- `structured_survey_clean.csv` (Qualtrics intake survey)
- `structured_participant_linkage.csv` (one row per participant, cross-file overlap + summary)

Findings this cleaning relies on (see `notebooks/15_structured_inspection.ipynb` for full detail):
- comma-delimited CSVs, no delimiter ambiguity
- `event_timestamp` / `task_start_at` / `task_end_at` need `format="mixed"` parsing
- `user_id` is already a shared canonical ID across both files — no prefix stripping needed
- `task_name` is 100% empty — dropped
- `update_type` mixes JSON-array-like strings with 2 bare-string values and a distinct `"[]"` value
- consent gating verified clean: no non-consenting participant has behavioral data


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

RAW = Path("../data_raw/structured/Retrospective")
BEH = RAW / "retro_behavioraldata_2026_07_22.csv"
SURV = RAW / "retro_surveydata_2026_07_22.csv"
OUT = Path("../outputs/cleaned")
OUT.mkdir(parents=True, exist_ok=True)
print("Paths configured")

## 1. Load Raw Files

In [ ]:
beh = pd.read_csv(BEH, low_memory=False)
surv_raw = pd.read_csv(SURV, low_memory=False)
print(f"Behavioral: {beh.shape}")
print(f"Survey (raw incl. Qualtrics header rows): {surv_raw.shape}")

## 2. Behavioral — Timestamps (UTC, mixed precision)

In [ ]:
beh['event_timestamp_utc'] = pd.to_datetime(beh['event_timestamp'], format='mixed', errors='coerce', utc=True)
beh['task_start_at_utc'] = pd.to_datetime(beh['task_start_at'], format='mixed', errors='coerce', utc=True)
beh['task_end_at_utc']   = pd.to_datetime(beh['task_end_at'], format='mixed', errors='coerce', utc=True)

assert beh['event_timestamp_utc'].isna().sum() == 0, "event_timestamp parse failures remain"
print("event_timestamp_utc parsed:", beh['event_timestamp_utc'].notna().sum(), "/", len(beh))
print("task_start_at_utc non-null:", beh['task_start_at_utc'].notna().sum(), "(timed tasks only, by design)")
print("Range:", beh['event_timestamp_utc'].min(), "to", beh['event_timestamp_utc'].max())

## 3. Behavioral — Parse `update_type` (mixed JSON-array / bare-string encoding)

Rules:
- `NaN` → not applicable (Created/Deleted events) → keep as `NaN` list
- `"[]"` → empty list, but a *distinct* value from NaN (an Updated event with no captured field changes)
- `'["a","b"]'` → JSON-parsed to `['a', 'b']`
- bare strings (`subtask_completed`, `subtask_uncompleted`) → wrapped as single-item list


In [ ]:
def parse_update_type(val):
    if pd.isna(val):
        return None
    val = val.strip()
    if val.startswith('['):
        try:
            return json.loads(val)
        except json.JSONDecodeError:
            return [val]
    return [val]

beh['update_type_parsed'] = beh['update_type'].apply(parse_update_type)
beh['update_type_list_str'] = beh['update_type_parsed'].apply(lambda x: '|'.join(x) if x is not None else np.nan)
beh['update_type_n_changes'] = beh['update_type_parsed'].apply(lambda x: len(x) if x is not None else np.nan)
beh['update_type_is_empty'] = beh['update_type_parsed'].apply(lambda x: x is not None and len(x) == 0)

print("Sample parsed values:")
print(beh[['update_type', 'update_type_list_str', 'update_type_n_changes']].drop_duplicates('update_type').head(12).to_string())
print()
print("Empty-array updates (event happened, no field flagged as changed):", beh['update_type_is_empty'].sum())

## 4. Behavioral — Drop Fully-Empty Column, Rename

In [ ]:
assert beh['task_name'].notna().sum() == 0
beh = beh.drop(columns=['task_name'])

beh = beh.rename(columns={
    'user_id': 'participant_id',
    'event': 'event_type',
    'task_duration': 'task_duration_min',
    'subscription_type': 'subscription_type_at_event',
})
print("task_name dropped (100% empty). Renamed: user_id->participant_id, event->event_type, "
      "task_duration->task_duration_min, subscription_type->subscription_type_at_event")

## 5. Behavioral — `days_since_first_event` per Participant

In [ ]:
first_ev = beh.groupby('participant_id')['event_timestamp_utc'].transform('min')
beh['days_since_first_event'] = (beh['event_timestamp_utc'] - first_ev).dt.total_seconds() / 86400
print(f"days_since_first_event: min={beh['days_since_first_event'].min():.2f}, max={beh['days_since_first_event'].max():.2f}")

## 6. Behavioral — Select Final Column Set

In [ ]:
BEH_COLS = [
    'participant_id', 'subscription_type_at_event',
    'event_type', 'update_type', 'update_type_list_str', 'update_type_n_changes', 'update_type_is_empty',
    'update_method',
    'event_timestamp_utc', 'event_timezone',
    'task_id', 'task_type', 'task_color', 'task_icon',
    'task_start_at_utc', 'task_end_at_utc', 'task_timezone', 'task_duration_min',
    'task_is_recurring', 'recurrence_unit', 'recurrence_interval',
    'task_has_notes', 'subtasks_amount',
    'days_since_first_event',
]
beh_clean = beh[BEH_COLS].rename(columns={'update_type': 'update_type_raw'}).sort_values(
    ['participant_id', 'event_timestamp_utc']).reset_index(drop=True)
print(beh_clean.shape)
beh_clean.head(3)

## 7. Behavioral — Validation

In [ ]:
assert beh_clean.duplicated().sum() == 0, "Duplicate rows found!"
assert beh_clean['participant_id'].notna().all()
assert beh_clean['event_timestamp_utc'].notna().all()
assert beh_clean['event_type'].isin(['Task Created', 'Task Updated', 'Task Deleted']).all()
assert (beh_clean.groupby('task_id')['participant_id'].nunique() <= 1).all(), "task_id shared across participants!"
print("All validation checks passed.")

## 8. Save Behavioral

In [ ]:
beh_clean.to_csv(OUT / "structured_behavioral_clean.csv", index=False)
print(f"Saved: structured_behavioral_clean.csv  ({beh_clean.shape})")

## 9. Survey — Drop Qualtrics Header Rows, Build Column Mapping

The question-text row (row 0 of the raw file) is preserved here as the empirical codebook, since no
separate codebook document exists for this survey.

In [ ]:
qtext = surv_raw.iloc[0].to_dict()
surv = surv_raw.iloc[2:].reset_index(drop=True)
print("Survey data rows:", len(surv))

COLUMN_MAP = {
    'Duration (in seconds)': 'duration_seconds',
    'user_id': 'participant_id',
    'Q12': 'consent',
    'Q21': 'age',
    'Q22': 'gender', 'Q22_5_TEXT': 'gender_other_text',
    'Q31': 'structured_use_duration',
    'Q311': 'plan_context', 'Q311_8_TEXT': 'plan_context_other_text',
    'Q312': 'usual_context', 'Q312_8_TEXT': 'usual_context_other_text',
    'Q34': 'app_goals', 'Q34_4_TEXT': 'app_goals_other_text',
    'Q41': 'life_happiness',
    'Q42': 'productivity_happiness',
    'Q43': 'pct_tasks_planned',
    'Q431': 'pct_planned_tasks_completed',
    'Q44': 'productivity_importance_general',
    'Q45': 'productivity_importance_personal',
    'Q46': 'productivity_rating_vs_others',
    'Q51': 'autism_status', 'Q51_4_TEXT': 'autism_other_text',
    'Q52': 'adhd_status', 'Q52_4_TEXT': 'adhd_other_text',
    'Q521_1': 'asrs_1_finish_details', 'Q521_2': 'asrs_2_organization',
    'Q521_3': 'asrs_3_remembering', 'Q521_4': 'asrs_4_avoid_delay',
    'Q521_5': 'asrs_5_fidget', 'Q521_6': 'asrs_6_overactive',
}
surv = surv.rename(columns=COLUMN_MAP)
print("Renamed", len(COLUMN_MAP), "columns. Remaining original names kept as-is:",
      [c for c in surv.columns if c not in COLUMN_MAP.values()])

## 10. Survey — Cast Metadata Fields

In [ ]:
surv['duration_seconds'] = pd.to_numeric(surv['duration_seconds'], errors='coerce')
surv['Progress'] = pd.to_numeric(surv['Progress'], errors='coerce')
surv['is_finished'] = surv['Finished'].map({'Wahr': True, 'Falsch': False})
surv['StartDate'] = pd.to_datetime(surv['StartDate'], errors='coerce')
surv['EndDate'] = pd.to_datetime(surv['EndDate'], errors='coerce')
surv['RecordedDate'] = pd.to_datetime(surv['RecordedDate'], errors='coerce')
surv['age'] = pd.to_numeric(surv['age'], errors='coerce')
surv['consent_given'] = surv['consent'] == 'I consent'
print(surv[['duration_seconds','Progress','is_finished','consent_given']].describe(include='all'))

## 11. Survey — Extract Numeric Grade from Likert Text (e.g. \"6 - happy\" → 6)

In [ ]:
def extract_leading_int(val):
    if pd.isna(val):
        return np.nan
    m = re.match(r'^\s*(\d+)', str(val))
    return int(m.group(1)) if m else np.nan

for col in ['life_happiness', 'productivity_happiness', 'productivity_importance_general',
            'productivity_importance_personal', 'productivity_rating_vs_others']:
    surv[col + '_score'] = surv[col].apply(extract_leading_int)

print(surv[['life_happiness', 'life_happiness_score']].drop_duplicates().head(8).to_string())

## 12. Survey — Percentage Fields to Numeric

In [ ]:
def extract_pct(val):
    if pd.isna(val):
        return np.nan
    m = re.match(r'^\s*(\d+)\s*%', str(val))
    return float(m.group(1)) if m else np.nan

surv['pct_tasks_planned_num'] = surv['pct_tasks_planned'].apply(extract_pct)
surv['pct_planned_tasks_completed_num'] = surv['pct_planned_tasks_completed'].apply(extract_pct)
print(surv[['pct_tasks_planned', 'pct_tasks_planned_num']].drop_duplicates().head(8).to_string())

## 13. Survey — ASRS-6 Ordinal Coding

Standard ASRS-v1.1 response-to-number mapping (Never=0 ... Very Often=4). This is a plain ordinal
recode of the response label, not the official clinical screen-positive scoring (which uses different
thresholds per item) — that construct-level scoring decision is deferred to a later modeling stage.

In [ ]:
ASRS_MAP = {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3, 'Very often': 4}
ASRS_COLS = ['asrs_1_finish_details', 'asrs_2_organization', 'asrs_3_remembering',
             'asrs_4_avoid_delay', 'asrs_5_fidget', 'asrs_6_overactive']

unmapped = set()
for col in ASRS_COLS:
    vals = set(surv[col].dropna().unique())
    unmapped |= (vals - set(ASRS_MAP.keys()))
assert not unmapped, f"Unmapped ASRS values found: {unmapped}"

for col in ASRS_COLS:
    surv[col + '_score'] = surv[col].map(ASRS_MAP)

print("ASRS items recoded 0-4. Non-null counts:")
print(surv[[c + '_score' for c in ASRS_COLS]].notna().sum())

## 14. Save Survey

In [ ]:
surv_clean = surv.reset_index(drop=True)
surv_clean.to_csv(OUT / "structured_survey_clean.csv", index=False)
print(f"Saved: structured_survey_clean.csv  ({surv_clean.shape})")

## 15. Participant Linkage

In [ ]:
beh_ids = set(beh_clean['participant_id'].unique())
surv_ids = set(surv_clean['participant_id'].unique())
all_ids = beh_ids | surv_ids

ev_counts = beh_clean.groupby('participant_id').agg(
    n_events=('event_type', 'count'),
    n_created=('event_type', lambda x: (x == 'Task Created').sum()),
    n_updated=('event_type', lambda x: (x == 'Task Updated').sum()),
    n_deleted=('event_type', lambda x: (x == 'Task Deleted').sum()),
    n_unique_tasks=('task_id', 'nunique'),
    days_observed=('days_since_first_event', 'max'),
    subscription_type_last=('subscription_type_at_event', 'last'),
)

linkage = pd.DataFrame({'participant_id': sorted(all_ids)})
linkage['in_behavioral'] = linkage['participant_id'].isin(beh_ids)
linkage['in_survey'] = linkage['participant_id'].isin(surv_ids)
linkage = linkage.merge(
    surv_clean[['participant_id', 'consent_given', 'is_finished']],
    on='participant_id', how='left')
linkage = linkage.merge(ev_counts, on='participant_id', how='left')

print(linkage.shape)
print()
print("in_behavioral & in_survey:", (linkage['in_behavioral'] & linkage['in_survey']).sum())
print("in_survey only:", (~linkage['in_behavioral'] & linkage['in_survey']).sum())
print("in_behavioral only (should be 0):", (linkage['in_behavioral'] & ~linkage['in_survey']).sum())

## 16. Save Participant Linkage

In [ ]:
linkage.to_csv(OUT / "structured_participant_linkage.csv", index=False)
print(f"Saved: structured_participant_linkage.csv  ({linkage.shape})")

## 17. Final Summary

In [ ]:
print(f"""
=== STRUCTURED RETROSPECTIVE — CLEANING COMPLETE ===

Outputs (outputs/cleaned/):
  structured_behavioral_clean.csv     : {beh_clean.shape[0]:,} rows x {beh_clean.shape[1]} cols
  structured_survey_clean.csv         : {surv_clean.shape[0]:,} rows x {surv_clean.shape[1]} cols
  structured_participant_linkage.csv  : {linkage.shape[0]:,} rows x {linkage.shape[1]} cols

Key transformations:
  - user_id kept as shared canonical participant_id (no prefix stripping needed, unlike DS5)
  - event_timestamp/task_start_at/task_end_at parsed with format="mixed" -> UTC
  - update_type parsed from mixed JSON-array/bare-string encoding -> list + count + is_empty flag
  - task_name dropped (100% empty)
  - Qualtrics 2-row header dropped from survey; Q-codes renamed to descriptive names (question text
    preserved as the empirical codebook in the companion .md)
  - Likert text ("6 - happy") -> numeric _score columns; percentage strings -> numeric
  - ASRS-6 items ordinally recoded 0-4 (Never..Very Often) -- NOT official clinical scoring
  - days_since_first_event derived per participant

Caveats carried into the cleaned files (see companion .md for full detail):
  - consent_given / is_finished flags are provided but rows are NOT deleted -- filter explicitly downstream
  - task_timezone still 96.7% missing (floating time) -- not imputed
  - ASRS 54.7% missing beyond base non-completion rate is unexplained (no skip-logic codebook available)
  - Updated/Deleted event schemas have no source codebook; documented from data alone
""")